# RMS Norm

## Summary

Root Mean Square Layer Normalization, or **RMSNorm**, is a simplified version of the standard Layer Normalization.

First off, I will explain the basics of RMSNorm, and then pros of using it, followed by technical remarks

**The Mathematical Formula**

Unlike LayerNorm, which shifts and scales the input, RMSNorm only scales the input based on the root mean square of the elements. For an input vector $x$, the operation is defined as:

$$
\bar{a_{i}} = \frac{x_i}{\text{RMS}(x)}g_i, \;\;\;\;\;\;\text{where}\;\;\; \text{RMS}(x)=\sqrt{\frac{1}{n}\sum_{i=1}^{n}{{x_i}^2}+\epsilon}
$$

**Notation**

* $x_i$: Input feature vector.
* $n$: The number of features (the dimension of the vector).
* $\epsilon$: A very small value to prevent division by zero which is often fixed `1e-5`.
* $g_i$: A learnable scaling parameter applied element-wise, allowing the model to adjust the signal's magnitude.


### Implimention

In [1]:
import math
import torch
from jaxtyping import Float
import torch.nn as nn
from torch.nn.parameter import Parameter

class RMSNorm(nn.Module):

    def __init__(
        self,
        d_model: int,           # size of the hidden dimension (e.g. 768)
        eps: float = 1e-5,      # small constant to prevent division by zero
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        
        factory_kwargs = {"device": device, "dtype": dtype}
        
        super().__init__()

        self.d_model = d_model
        self.eps = eps

        # γ (gamma) — learned scale vector, one value per feature dimension
        # shape: (d_model,)  e.g. (768,)
        # unlike Linear, this is a 1D vector not a matrix — just a per-feature scale
        self.weight = Parameter(
            torch.empty(self.d_model, **factory_kwargs)
        )
        self.reset_parameters()
    
    def reset_parameters(self) -> None:
        # initialize γ with truncated normal distribution
        # (small random values near 0, not all-ones like typical LayerNorm init)
        nn.init.trunc_normal_(self.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # remember original dtype (could be float16/bfloat16 in real LLMs)
        in_dtype = x.dtype

        # cast to float32 for numerical stability during normalization
        # normalization math is sensitive to precision — fp16 can cause NaNs here
        x = x.to(torch.float32)

        # compute RMS across the feature dimension (last dim)
        # x**2          → square each element
        # mean(..., dim=-1) → average across features, shape: (batch, seq, 1)
        # keepdim=True  → keep the dim so broadcasting works in division below
        # + self.eps    → avoid sqrt(0) which would cause division by zero
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)

        # normalize: divide each vector by its own RMS magnitude
        # this makes all vectors have RMS ≈ 1.0
        x_norm = x / rms
        
        # apply learned scale γ (self.weight)
        # self.weight shape: (d_model,) — broadcasts across batch and sequence dims
        # this lets the model learn "how much" each feature should be scaled
        result = x_norm * self.weight
        
        # cast back to original dtype (e.g. bfloat16) before returning
        return result.to(in_dtype)

In [2]:
# ── 1. Setup dimensions ───────────────────────────────────────────────────────
batch_size = 2
seq_len    = 3      # number of tokens in the sequence
d_model    = 4      # hidden dimension size

# ── 2. Instantiate RMSNorm ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rms_norm = RMSNorm(d_model=d_model, device=device)

# ── 3. Create a dummy input tensor ────────────────────────────────────────────
# Shape: (batch_size, seq_len, d_model)
# In a real transformer this would be the output of an attention or FFN layer
x = torch.tensor([
    # batch 0
    [[ 2.0,  4.0,  6.0,  8.0],   # token 0
     [ 1.0, -1.0,  2.0, -2.0],   # token 1
     [ 0.5,  0.5,  0.5,  0.5]],  # token 2
    # batch 1
    [[ 3.0,  0.0, -3.0,  6.0],   # token 0
     [-1.0, -1.0, -1.0, -1.0],   # token 1
     [ 4.0,  2.0,  0.0, -2.0]],  # token 2
], dtype=torch.float32).to(device)

# ── 4. Run forward pass ───────────────────────────────────────────────────────
output = rms_norm(x)

# ── 5. Print results ──────────────────────────────────────────────────────────
print(f"Input shape:  {x.shape}")
print("Input tensor:")
print(x)

print(f"\nγ (weight) shape: {rms_norm.weight.shape}")
print("γ (weight):")
print(rms_norm.weight)

print(f"\nOutput shape: {output.shape}")
print("Output tensor:")
print(output)

# ── 6. Manual verification for batch=0, token=0 ───────────────────────────────
print("\n── Manual check: batch 0, token 0 ──")
token = torch.tensor([2.0, 4.0, 6.0, 8.0])
rms   = torch.sqrt(torch.mean(token**2) + 1e-5)
norm  = token / rms
print(f"  token:    {token.tolist()}")
print(f"  RMS:      {rms.item():.6f}")
print(f"  x / RMS:  {norm.tolist()}")
print(f"  * gamma:  {(norm * rms_norm.weight.cpu()).tolist()}")
print(f"  (matches output[0][0]: {output[0][0].tolist()})")

Input shape:  torch.Size([2, 3, 4])
Input tensor:
tensor([[[ 2.0000,  4.0000,  6.0000,  8.0000],
         [ 1.0000, -1.0000,  2.0000, -2.0000],
         [ 0.5000,  0.5000,  0.5000,  0.5000]],

        [[ 3.0000,  0.0000, -3.0000,  6.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000],
         [ 4.0000,  2.0000,  0.0000, -2.0000]]])

γ (weight) shape: torch.Size([4])
γ (weight):
Parameter containing:
tensor([ 0.7360, -0.0975,  0.8145, -1.1483], requires_grad=True)

Output shape: torch.Size([2, 3, 4])
Output tensor:
tensor([[[ 0.2688, -0.0712,  0.8923, -1.6772],
         [ 0.4655,  0.0617,  1.0303,  1.4525],
         [ 0.7360, -0.0975,  0.8145, -1.1483]],

        [[ 0.6010, -0.0000, -0.6651, -1.8751],
         [-0.7360,  0.0975, -0.8145,  1.1483],
         [ 1.2019, -0.0796,  0.0000,  0.9376]]], grad_fn=<MulBackward0>)

── Manual check: batch 0, token 0 ──
  token:    [2.0, 4.0, 6.0, 8.0]
  RMS:      5.477226
  x / RMS:  [0.3651483356952667, 0.7302966713905334, 1.0954450368881226, 1.4

## Details

### Pre-Norm vs Post-Norm

**Learning Stability `pre-norm` > `post-norm`**

<img src="../images/PreNormPostNorm.png" width="70%">

This paragraph is describing **where LayerNorm is placed** inside a Transformer block.


**1.Post-Norm (Original Transformer, 2017)**

In the original paper (Vaswani et al.), they did:

```
x → F(x) → + → LayerNorm → output
      ↑
      x (residual)
```



So mathematically:

$$
y = \text{LayerNorm}(x + F(x))
$$

<!-- Then:

$$
x = \text{LayerNorm}(x + \text{MLP}(x))
$$
 -->

---

**2 Pre-Norm (Modern Transformers)**

Now modern models do this instead:

```
x → LayerNorm → F(x) → +
↑                      ↓
└─────── residual ─────┘
```

So mathematically:

$$
y = x + F(\text{LayerNorm}(x))
$$

<!-- Then:

$$
x = x + \text{MLP}(\text{LayerNorm}(x))
$$ -->

Now normalization happens **before** the sublayer.

---

**Why is Pre-Norm better?**

The key idea:

In pre-norm, there is a **clean residual path**. **This helps gradients flow better!!** -> why?

But first of all, **What does “clean residual path” mean?**

In Pre-Norm:

$$
y = x + F(x)
$$

So derivative w.r.t. x:

$$
\frac{dy}{dx} = 1 + \frac{dF}{dx}
$$

There is always a direct identity term:

$$
\frac{dy}{dx} \supset 1
$$

if $\frac{dF}{dx} \approx 0$. This is the key.

**Why this helps gradient flow?**

When backpropagating:

$$
\frac{dL}{dx} = \frac{dL}{dy} \cdot \frac{dy}{dx}\;\;\;\;\;\; \text{where}\;\;\; L: \text{loss function}
$$

If:

$$
\frac{dy}{dx} = 1 + \text{small term}
$$

Then:

$$
\frac{dL}{dx} \approx \frac{dL}{dy}
$$

Gradient can pass almost unchanged.

So even if:

* F(x) saturates
* F(x) has small gradients
* F(x) has exploding gradients

The identity path guarantees:


### Upcast to `torch.float32`

We should know some techniques to prevent overflow when we square the input

**🔎 Why “upcast to `torch.float32` ”?**

If your input tensor `x` is:

* `torch.float16` (fp16), or
* `torch.bfloat16`

and you compute:

```python
x ** 2
```

you might get **numerical overflow** because these formats have much smaller numeric ranges and lower precision than `float32`.

> Convert the tensor to `float32` before doing math operations like squaring.

**🧠 What does “upcast” mean?**

**Upcast** = convert to a higher precision type.

Your `forward` method shold look like:

```python
in_dtype = x.dtype
x = x.to(torch.float32)
# Your code here performing RMSNorm
...
result =
...
# Return the result in the original dtype
return result.to(in_dtype)
```
